## Code Availability Notice

This notebook is provided to demonstrate the implementation and overall analysis workflow used in the study. Data paths, application identifiers, and selected values have been omitted or anonymized for privacy.

The code is not intended as a fully executable replication package. Results may vary depending on the data version, preprocessing environment, library versions, and random seed.

In [ ]:
!pip install pandas numpy scipy gensim google-play-scraper
!pip install konlpy kiwipiepy
!pip install git+https://github.com/haven-jeon/PyKoSpacing.git
!pip install bertopic hdbscan
!pip install --upgrade sentence-transformers
!pip install --upgrade tensorflow==2.16.2 h5py==3.10.0

In [ ]:
import pandas as pd
import numpy as np
import pandas as pd
import ast
import umap
import matplotlib.pyplot as plt
from google_play_scraper import Sort, reviews_all
from sentence_transformers import SentenceTransformer

## STEP 1. Data Preparation

### Selecting crawling Channels & developing code + Data Collection

In [ ]:
app_name = " "
target_year = 2023

app_operation = reviews_all(
    app_name,
    sleep_milliseconds=20,
    lang="ko",
    country="kr",
    sort = Sort.NEWEST,
)

reviews_list = []

for review in app_operation:
    review_date = review['at']
    if review_date.year >= target_year:
        review_dict = {
            'reviewId': review['reviewId'],
            'userName': review['userName'],
            'review': review['content'],
            'score': review['score'],
            'thumbs_up_count': review['thumbsUpCount'],
            'version': review['reviewCreatedVersion'],
            'date': review_date,
        }
        reviews_list.append(review_dict)

reviews_df = pd.DataFrame(reviews_list) #until 2025.04.12 00:00
#reviews_df_filtered = reviews_df[reviews_df['date'].between('2025-02-04','2025-04-12')]
reviews_df.info()

## STEP 2. Data Preprocessing

In [ ]:
import re
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from tqdm.notebook import tqdm
tqdm.pandas()

from pykospacing import Spacing
from kiwipiepy import Kiwi

import gensim
from gensim.models import Phrases
from gensim.models.phrases import Phraser

spacing = Spacing()
kiwi = Kiwi()


In [ ]:
# Add the app identifier and pass the collected reviews to preprocessing.
reviews_df['AppName'] = app_name
totalData = reviews_df.copy()


In [ ]:
strColumns = ["AppName", "reviewId", "review"]
intColumn = ["score"]

for i in range(len(strColumns)):
    totalData[strColumns[i]] = totalData[strColumns[i]].astype(str)

for i in range(len(intColumn)):
    totalData[intColumn[i]] = totalData[intColumn[i]].astype(int)
totalData['date'] = pd.to_datetime(totalData['date'])

totalData.info()

In [ ]:
def clean_text(text): #Only Koreans
    cleaned = re.sub(r'[^가-힣\s]', '', text)
    return cleaned
totalData["review"] = totalData["review"].apply(clean_text)

def apply_spacing(text): #Spacing Tools
     spacing = Spacing()
     spaced_text = spacing(text)
     return spaced_text

totalData["review"] = totalData["review"].apply(apply_spacing)
totalData 

In [ ]:
dup_reviews = totalData[totalData["review"].duplicated()]
dup_reviews.info()

In [ ]:
totalData = totalData.drop_duplicates(subset=["review"])
totalData.info()

In [ ]:
totalData = totalData[totalData["review"].str.len() >= 11]
totalData

In [ ]:
# Colab path for the 1,309-word stopword dictionary used in this project.
STOPWORDS_PATH = "/content/stopwords.csv"
stopwords_df = pd.read_csv(STOPWORDS_PATH)
stopwords = set(stopwords_df["stopwords"].dropna().astype(str))
print(f"Loaded {len(stopwords):,} stopwords")


In [ ]:
KIWI_ALLOWED_TAGS = {
    "NNG",  # common noun
    "NNP",  # proper noun
    "VV",   # verb
    "VA",   # adjective
    "VX",   # auxiliary predicate
    "VCP",  # positive copula
    "VCN",  # negative copula
    "MAG",  # general adverb
}

def preprocessing(string):
    string = kiwi.space(str(string))
    string = re.sub(r'(?:안녕하세요|감사합니다|감사해요|감사드려요)', ' ', string)
    string = re.sub(r'\s+', ' ', string).strip()

    tokens = [
        token.form
        for token in kiwi.tokenize(string)
        if token.tag in KIWI_ALLOWED_TAGS
        and token.form not in stopwords
    ]

    return tokens, tokens


def bi_trigram(data_words):
    bigram = gensim.models.Phrases(data_words, min_count=3, threshold=100)
    trigram = gensim.models.Phrases(bigram[data_words], threshold=100)

    bigram_mod = gensim.models.phrases.Phraser(bigram)
    trigram_mod = gensim.models.phrases.Phraser(trigram)

    texts = [bigram_mod[doc] for doc in data_words]
    texts = [trigram_mod[doc] for doc in texts]
    return texts


In [ ]:
totalData[['Kiwi_token_raw', 'pos_tagged']] = totalData['review'].progress_apply(lambda x: pd.Series(preprocessing(x)))

In [ ]:
# bi/tri-gram Apply
tokens_list = totalData['pos_tagged'].tolist()

# Generate bi/tri-gram for all tokens
tokens_bi_tri = bi_trigram(tokens_list)

# Save back to totalData
totalData["Kiwi_token"] = tokens_bi_tri

# Eliminate rows with empty Kiwi_token
totalData = totalData[totalData['Kiwi_token'].apply(lambda x: len(x) > 0)]
totalData.reset_index(drop=True, inplace=True)

totalData.head()

In [ ]:
# If data is NaN or not a list, filter and check
problem_rows = totalData[totalData['Kiwi_token'].isna() | totalData['Kiwi_token'].apply(lambda x: not isinstance(x, list))]
print(f"Problem rows: {len(problem_rows)}")
problem_rows.head()

In [ ]:
totalData.to_csv(". .csv", encoding = 'utf-8-sig', index = False)

## STEP 3. Topic Modeling (BERTopic)

In [ ]:
import ast
import pandas as pd
import numpy as np
import umap
import matplotlib.pyplot as plt
from bertopic import BERTopic
from hdbscan import HDBSCAN
from sentence_transformers import SentenceTransformer


### 3-1. Load processed data and create embeddings

In [ ]:
finalDf = pd.read_csv("/content/kiwi_전처리v2_0430 (1).csv")

def restore_token_list(value):
    if isinstance(value, list):
        return value
    return ast.literal_eval(value)

finalDf["Kiwi_token"] = finalDf["Kiwi_token"].apply(restore_token_list)


In [ ]:
model = SentenceTransformer('snunlp/KR-SBERT-V40K-klueNLI-augSTS')

In [ ]:
# docs 생성
docs = finalDf['Kiwi_token'].apply(lambda x: ' '.join(x) if isinstance(x, list) else x).tolist()

# 모델에 텍스트를 입력하여 임베딩 계산
sentence_embeddings = model.encode(docs)

The embeddings are generated from `docs` after loading and restoring `Kiwi_token`.

### 3-2. Dimension reduction and clustering (UMAP + HDBSCAN)

In [ ]:
umap_emb = umap.UMAP(n_neighbors=15,
                     min_dist=0.0,
                     n_components=5,
                     metric='cosine')


#HDBSCAN 모델 정의
hdbscan_model = HDBSCAN(min_cluster_size=15, # 더 작게
                        min_samples=5, # 더 관대하게
                        metric='euclidean',
                        prediction_data=True)


# BERTopic에 UMAP과 HDBSCAN 모델 넘기기
topic_model = BERTopic(umap_model=umap_emb,
                       hdbscan_model=hdbscan_model,
                       language="ko",
                       calculate_probabilities=True)

In [ ]:
topics, probs = topic_model.fit_transform(docs, sentence_embeddings)

topic_model = topic_model.reduce_topics(docs, nr_topics=20)

topics, probs = topic_model.transform(docs, embeddings=sentence_embeddings)
topic_model.probabilities_ = probs

In [ ]:
# Attach the reduced topic and probability vector to each source review.
finalDf["Reduced_Topic"] = topics
finalDf["Probability"] = [prob.tolist() for prob in probs]
finalDf.to_csv("bertopic_results.csv", encoding="utf-8-sig", index=False)


### 3-3. Topic extraction and inspection (c-TF-IDF)

In [ ]:
for i, (topic, prob) in enumerate(zip(topics, probs)):
    print(f"Review {i}: Topic - {topic}, Probability - {prob}")


In [ ]:
fig = topic_model.visualize_topics()
fig.show()

In [ ]:
fig_bar = topic_model.visualize_barchart(top_n_topics=20)
fig_bar.show()

## STEP 4. Statistical Analysis

### 4-1. Regression Analysis

In [ ]:
import ast
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.outliers_influence import variance_inflation_factor


In [ ]:
path = "  "
df = pd.read_csv(path)
df.info()

In [ ]:
df["superAppYN"] = np.where(df["AppName"] == "  ", 1, 0)

def parse_prob_string(prob_str):
    if isinstance(prob_str, list):
        return prob_str
    return ast.literal_eval(prob_str)

In [ ]:
# Apply to all Data
df["prob_list"] = df["Probability"].apply(parse_prob_string)

# Topic Column Decompostition
topic_probs = pd.DataFrame(df["prob_list"].tolist(), columns=[f"topic_{i}" for i in range(len(df['prob_list'][0]))])

# Merging Data for Regression Analysis
regression_df = pd.concat([df["score"], topic_probs], axis=1).dropna()

In [ ]:
# 1) Statistics Linear Algebra Extract
P = pd.DataFrame(
    topic_model.probabilities_,
    columns=[f"Topic_{k}" for k in range(topic_model.probabilities_.shape[1])]
)

# Choosing imporant topics based on mean probability
K_keep = 14
topk = P.mean().nlargest(K_keep).index        # 발생비중 기준
P = P[topk]

# Correlation Heatmap
sns.heatmap(P.corr(), cmap='coolwarm', center=0)
plt.title("Topic Correlation"); plt.show()

# Variance Inflation Factor (VIF) Calculation
X_const = np.column_stack([np.ones(len(P)), P.values])  # const 포함
vif = pd.Series(
    [variance_inflation_factor(X_const, i) for i in range(X_const.shape[1])],
    index=['const'] + list(P.columns)
)
print("\nTop VIF")
print(vif.sort_values(ascending=False).head(10))

#### 1st Regression Analysis

In [ ]:
X = regression_df.drop(columns="score")
y = df["score"]

model = sm.OLS(y, X).fit()
print(model.summary())

In [ ]:
summary_df1 = pd.DataFrame({
    'Coefficient': model.params,
    'Standard Error': model.bse,
    'T-statistic': model.tvalues,
    'P-value': model.pvalues,
    'Confidence Interval (Lower)': model.conf_int()[0],
    'Confidence Interval (Upper)': model.conf_int()[1]
})

summary_df1.to_csv(' .csv') # Save the DataFrame to a CSV file

#### 2nd Regression Analysis

In [ ]:
X = pd.concat([regression_df.drop(columns="score"), df["superAppYN"]], axis=1)
y = df["score"]

model = sm.OLS(y, X).fit()
print(model.summary())

In [ ]:
summary_df2 = pd.DataFrame({
    'Coefficient': model.params,
    'Standard Error': model.bse,
    'T-statistic': model.tvalues,
    'P-value': model.pvalues,
    'Confidence Interval (Lower)': model.conf_int()[0],
    'Confidence Interval (Upper)': model.conf_int()[1]
})

# Save the DataFrame to a CSV file
summary_df2.to_csv('  .csv')

#### 3rd Regression Analysis

In [ ]:
# Interaction Index (Topic Possibility * SuperApp Y/N)
interaction_terms = regression_df.drop(columns="score").multiply(df["superAppYN"], axis=0)
interaction_terms.columns = [col + '_x_superAppYN' for col in interaction_terms.columns]

X = pd.concat([regression_df.drop(columns="score"), df["superAppYN"], interaction_terms], axis=1)
y = df["score"]

model = sm.OLS(y, X).fit()
print(model.summary())

#### Export 3rd regression results

In [ ]:
summary_df3 = pd.DataFrame({
    'Coefficient': model.params,
    'Standard Error': model.bse,
    'T-statistic': model.tvalues,
    'P-value': model.pvalues,
    'Confidence Interval (Lower)': model.conf_int()[0],
    'Confidence Interval (Upper)': model.conf_int()[1]
})

summary_df3.to_csv('. .csv')

### 4-2. Multigroup Regression Analysis

#### Dividing SuperApp and Non-SuperApp for Regression Analysis

In [ ]:
df1 = df.loc[df["superAppYN"]== 1]
df1.head(3)

In [ ]:
X = regression_df.loc[df["superAppYN"] == 1].drop(columns="score")
y = df.loc[df["superAppYN"] == 1, "score"]

model = sm.OLS(y, X).fit()
print(model.summary())

In [ ]:
print('Pvalue :', model.pvalues)

In [ ]:
summary_SuperApp1 = pd.DataFrame({
    'Coefficient': model.params,
    'Standard Error': model.bse,
    'T-statistic': model.tvalues,
    'P-value': model.pvalues,
    'Confidence Interval (Lower)': model.conf_int()[0],
    'Confidence Interval (Upper)': model.conf_int()[1]
})

# Save the DataFrame to a CSV file
summary_SuperApp1.to_csv('. .csv')

In [ ]:
df2 = df.loc[df["superAppYN"]!= 1]
df2.info()

In [ ]:
X = regression_df.loc[df["superAppYN"] == 0].drop(columns="score")
y = df.loc[df["superAppYN"] == 0, "score"]

model = sm.OLS(y, X).fit()
print(model.summary())

In [ ]:
print('Pvalue :', model.pvalues)

In [ ]:
summary_SuperApp0 = pd.DataFrame({
    'Coefficient': model.params,
    'Standard Error': model.bse,
    'T-statistic': model.tvalues,
    'P-value': model.pvalues,
    'Confidence Interval (Lower)': model.conf_int()[0],
    'Confidence Interval (Upper)': model.conf_int()[1]
})

# Save the DataFrame to a CSV file
summary_SuperApp0.to_csv('. .csv')

### 4-3. Proportion Analysis 

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.io as pio
import numpy as np
import seaborn as sns
from statsmodels.stats.outliers_influence import variance_inflation_factor

plt.rc('font', family='NanumGothic')
%matplotlib inline

In [ ]:
df = pd.read_csv(". .csv")

#df.drop(columns=['Topic'], inplace=True) 
df.rename(columns={'Reduced_Topic': 'Topic'}, inplace=True)
#df = df[df['Topic'] != -1] 
df.info()

In [ ]:
df['date'] = pd.to_datetime(df['date'])
df['year_month'] = df['date'].dt.to_period('M').astype(str)

In [ ]:
grouped = df.groupby(['AppName', 'year_month', 'Topic']).size().reset_index(name='review_count')
total_reviews = grouped.groupby(['AppName', 'year_month'])['review_count'].sum().reset_index(name='total_reviews')
grouped = pd.merge(grouped, total_reviews, on=['AppName', 'year_month'])
grouped['review_ratio'] = (grouped['review_count'] / grouped['total_reviews']) * 100

In [ ]:
pivot_table = grouped.pivot_table(index='year_month', columns=['AppName', 'Topic'], values='review_ratio', fill_value=0)
pivot_table.index = pd.to_datetime(pivot_table.index, format='%Y-%m')
#pivot_table = pivot_table[pivot_table.index >= '2023-01-01']
baseline_date = pd.to_datetime('2023-12-01')

In [ ]:
app_list = ['A', 'B', 'C', 'D', 'E']  

APP_COLORS = {  
    'InsuranceApp': '#cc79a7',
    'BankingApp': '#009e73',
    'SecuritiesApp': '#0072b2',
    'PaymentApp': '#e69f00',
    'SuperApp': '#d55e00'
}

APP_DISPLAY_NAMES = {
    'A': 'InsuranceApp',
    'B': 'BankingApp',
    'C': 'SecuritiesApp',
    'D': 'PaymentApp',
    'E': 'SuperApp'}

BASELINE_DATE = pd.to_datetime('2023-12-01')


df['date'] = pd.to_datetime(df['date'])
df['year_month'] = df['date'].dt.to_period('M').dt.to_timestamp()

grouped = (
    df.groupby(['AppName','year_month','Topic'])
      .size()
      .reset_index(name='review_count')
)
grouped['total_reviews'] = (
    grouped.groupby(['AppName','year_month'])['review_count']
           .transform('sum')
)
grouped['review_ratio'] = grouped['review_count'] / grouped['total_reviews'] * 100
long_df = grouped.rename(columns={'review_ratio':'share'})



long_df['AppName'] = long_df['AppName'].map(APP_DISPLAY_NAMES)


# 2. Making Individual Graphs per Topic
topics = sorted(long_df['Topic'].unique())

for topic in topics:
    df_topic = long_df[long_df['Topic'] == topic]

    fig = px.line(
        df_topic,
        x='year_month', y='share',
        color='AppName',                    
        color_discrete_map=APP_COLORS,
        markers=True,
        title=f'Topic {topic} Monthly Review Share by App'
    )

    fig.update_yaxes(range=[0,10], title='Share (%)')

    fig.add_vline(
        x=BASELINE_DATE,
        line_color='red', line_dash='dash', line_width=2
    )
    fig.add_annotation(
        x=BASELINE_DATE,
        y=0, yref='paper',
        text='2023-12',
        showarrow=False, yshift=320,
        font=dict(color='red', size=12)
    )

    fig.update_layout(
        legend_title_text='App',
        height=500,
        margin=dict(t=10, b=40, l=20, r=40),
        plot_bgcolor='white',
        legend=dict(
            x=1, y=1,
            xanchor ='right', yanchor='bottom',
            bgcolor='rgba(0,0,0,0)',
            font=dict(size=14)
        ),
        title_font=dict(size=18)
    )
    fig.update_traces(line=dict(width=4))
    fig.update_xaxes(gridcolor='rgba(0,0,0,0.1)', gridwidth=1)
    fig.show()                                   
    fig.write_html(f'topic_{topic}_share.html')  